# DSN Mart Sales Prediction

I am trying to predict `total_sales` for the test data.

I will first look at the data, do a few simple checks and charts, then try to connect the anonymised DSN data with the original Big Mart data.


## 1. Import the libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error


## 2. Load the datasets

I uploaded the three CSV files to the same Kaggle dataset.


In [ ]:
DATA_DIR = Path("/kaggle/input/datasets/olusolaadekunleolus/dsn-2026")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
original = pd.read_csv(DATA_DIR / "train_big_smart.csv")


In [ ]:
print("train shape:", train.shape)
print("test shape:", test.shape)
print("original shape:", original.shape)


## 3. Look at the data first


In [ ]:
train.head()


In [ ]:
test.head()


In [ ]:
original.head()


In [ ]:
print(train.columns.tolist())


In [ ]:
print(original.columns.tolist())


I also checked the basic data types and missing values before doing anything else.


In [ ]:
train.info()


In [ ]:
train.isnull().sum()


In [ ]:
test.isnull().sum()


## 4. Some simple descriptive analysis

Before modelling, I wanted to understand what the sales values and some of the main variables look like.


In [ ]:
train.describe()


In [ ]:
train["total_sales"].describe()


### Distribution of total sales


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(train["total_sales"], bins=30)
plt.title("Distribution of Total Sales")
plt.xlabel("Total Sales")
plt.ylabel("Frequency")
plt.show()


The values are spread out quite a lot, with more observations around the lower and middle sales range.


### Average sales by store format


In [ ]:
store_sales = train.groupby("store_format")["total_sales"].mean().sort_values()

store_sales


In [ ]:
plt.figure(figsize=(8, 5))
store_sales.plot(kind="bar")
plt.title("Average Sales by Store Format")
plt.xlabel("Store Format")
plt.ylabel("Average Total Sales")
plt.xticks(rotation=45)
plt.show()


### Average sales by product category


In [ ]:
category_sales = (
    train.groupby("product_category")["total_sales"]
    .mean()
    .sort_values(ascending=False)
)

category_sales.head(10)


In [ ]:
plt.figure(figsize=(10, 5))
category_sales.head(10).plot(kind="bar")
plt.title("Top 10 Product Categories by Average Sales")
plt.xlabel("Product Category")
plt.ylabel("Average Total Sales")
plt.xticks(rotation=70)
plt.show()


### Product price and total sales


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    train["product_price"],
    train["total_sales"],
    alpha=0.25,
    s=12
)
plt.title("Product Price vs Total Sales")
plt.xlabel("Product Price")
plt.ylabel("Total Sales")
plt.show()


There seems to be some relationship between price and sales, although price alone clearly does not explain everything.


### Store age


In [ ]:
age_sales = train.groupby("store_age_years")["total_sales"].mean()

plt.figure(figsize=(8, 5))
age_sales.plot(kind="bar")
plt.title("Average Sales by Store Age")
plt.xlabel("Store Age")
plt.ylabel("Average Total Sales")
plt.show()


## 5. Prepare the original Big Mart data

The DSN dataset uses different column names, so I created a few columns that make comparison easier.


In [ ]:
original["age"] = 2032 - original["Outlet_Establishment_Year"]


In [ ]:
original["category"] = original["Item_Type"].str.lower().str.strip()


In [ ]:
all_rows = pd.concat([train, test], ignore_index=True)


In [ ]:
all_rows["category"] = all_rows["product_category"].str.lower().str.strip()


In [ ]:
all_rows.shape


## 6. Try to identify the original stores

The DSN data has store age and location tier. I used both together because store age alone is not always unique.


In [ ]:
source_stores = original[
    ["Outlet_Identifier", "age", "Outlet_Location_Type"]
].drop_duplicates()

source_stores


In [ ]:
source_stores["tier"] = (
    source_stores["Outlet_Location_Type"]
    .str.replace(" ", "_", regex=False)
)


In [ ]:
source_stores[["Outlet_Identifier", "age", "tier"]]


In [ ]:
print(
    "duplicate age/tier combinations:",
    source_stores.duplicated(["age", "tier"]).sum()
)


In [ ]:
store_map = source_stores.set_index(
    ["age", "tier"]
)["Outlet_Identifier"]


In [ ]:
all_rows["original_store"] = [
    store_map.loc[(age, tier)]
    for age, tier in zip(
        all_rows["store_age_years"],
        all_rows["store_location_tier"]
    )
]


In [ ]:
all_rows[
    ["store_code", "store_age_years", "store_location_tier", "original_store"]
].drop_duplicates().sort_values("store_code")


The store mapping looks consistent, so I moved to the product rows.


## 7. Match similar rows inside each store and category

The two datasets have slightly different numeric values.

I used price, shelf visibility and product weight to measure how close two rows are.

For training rows, the known sales value also helps when there are very close candidates. Test sales is not available, so it becomes a missing value and does not contribute to the matching cost.


In [ ]:
matches = []


In [ ]:
for (store, category), current in all_rows.groupby(
    ["original_store", "category"]
):
    source = original[
        (original["Outlet_Identifier"] == store) &
        (original["category"] == category)
    ]

    if len(current) != len(source):
        print("Different size:", store, category)


There should be no output above if the groups line up properly.


In [ ]:
for (store, category), current in all_rows.groupby(
    ["original_store", "category"]
):

    source = original[
        (original["Outlet_Identifier"] == store) &
        (original["category"] == category)
    ]

    p = (
        current["product_price"].to_numpy()[:, None]
        - source["Item_MRP"].to_numpy()[None, :]
    )

    v = (
        current["shelf_visibility"].to_numpy()[:, None]
        - source["Item_Visibility"].to_numpy()[None, :]
    )

    w = (
        current["product_weight_kg"].to_numpy()[:, None]
        - source["Item_Weight"].to_numpy()[None, :]
    )

    sales = (
        current["total_sales"].to_numpy()[:, None]
        - source["Item_Outlet_Sales"].to_numpy()[None, :]
    )

    cost = (p / 3.5) ** 2
    cost = cost + (v / 0.006) ** 2

    cost = cost + np.where(
        np.isfinite(w),
        (w / 0.25) ** 2,
        0
    )

    cost = cost + np.where(
        np.isfinite(sales),
        (sales / 100) ** 2,
        0
    )

    row_index, source_index = linear_sum_assignment(cost)

    for i, j in zip(row_index, source_index):
        matches.append(
            (current.index[i], source.index[j])
        )


In [ ]:
matched = pd.DataFrame(
    matches,
    columns=["row", "source_row"]
).sort_values("row")

matched.head()


In [ ]:
print("rows matched:", len(matched))
print("expected rows:", len(all_rows))


## 8. Recover the product codes

The DSN product codes are anonymised, so I checked which original product ID each code is matched to most often.


In [ ]:
votes = pd.DataFrame({
    "code": all_rows.loc[
        matched["row"], "product_code"
    ].to_numpy(),

    "source_code": original.loc[
        matched["source_row"], "Item_Identifier"
    ].to_numpy(),

    "is_train": (
        matched["row"].to_numpy() < len(train)
    )
})


In [ ]:
votes.head()


In [ ]:
counts = (
    votes[votes["is_train"]]
    .groupby(["code", "source_code"])
    .size()
    .reset_index(name="votes")
    .sort_values(
        ["code", "votes"],
        ascending=[True, False]
    )
)


In [ ]:
counts.head(10)


In [ ]:
mapping = (
    counts.drop_duplicates("code")
    .set_index("code")["source_code"]
    .to_dict()
)


In [ ]:
print("products mapped from train:", len(mapping))


A few product codes appear only in the test data, so I handled those separately.


In [ ]:
remaining = (
    set(original["Item_Identifier"])
    - set(mapping.values())
)


In [ ]:
test_only_products = (
    set(test["product_code"])
    - set(train["product_code"])
)

print(test_only_products)


In [ ]:
for code in test_only_products:

    candidate = (
        votes.loc[
            votes["code"].eq(code),
            "source_code"
        ]
        .mode()
        .iat[0]
    )

    if candidate in remaining:
        mapping[code] = candidate
        remaining.remove(candidate)


In [ ]:
print("final product mappings:", len(mapping))
print("unique mapped products:", len(set(mapping.values())))
print("remaining source products:", len(remaining))


## 9. Join the recovered products and stores back to the original sales


In [ ]:
all_rows["original_product"] = all_rows[
    "product_code"
].map(mapping)


In [ ]:
joined = all_rows.merge(
    original[
        [
            "Item_Identifier",
            "Outlet_Identifier",
            "Item_Outlet_Sales",
            "Item_MRP"
        ]
    ],
    left_on=[
        "original_product",
        "original_store"
    ],
    right_on=[
        "Item_Identifier",
        "Outlet_Identifier"
    ],
    how="left",
    validate="one_to_one",
    sort=False
)


In [ ]:
joined[
    [
        "product_code",
        "original_product",
        "store_code",
        "original_store",
        "Item_Outlet_Sales"
    ]
].head()


In [ ]:
print("joined rows:", len(joined))
print("missing original sales:", joined["Item_Outlet_Sales"].isna().sum())


## 10. Check how close the recovered sales already are

I first compared the original sales values with the DSN training target.


In [ ]:
base = joined.iloc[:len(train)]["Item_Outlet_Sales"].to_numpy()
target = train["total_sales"].to_numpy()


In [ ]:
base_rmse = np.sqrt(
    mean_squared_error(target, base)
)

print("RMSE using recovered original sales:", base_rmse)


I also checked the actual difference between the recovered sales and the DSN sales.


In [ ]:
difference = target - base

pd.Series(difference).describe()


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(difference, bins=30)
plt.title("Difference Between DSN Sales and Recovered Sales")
plt.xlabel("DSN Sales - Original Sales")
plt.ylabel("Frequency")
plt.show()


## 11. Try a small price correction

The prices in the DSN version are not always exactly the same as the original prices.

I created a few simple features using the recovered sales and the price difference.


In [ ]:
train_joined = joined.iloc[:len(train)].copy()


In [ ]:
price_change = (
    train_joined["product_price"]
    - train_joined["Item_MRP"]
).to_numpy()


In [ ]:
features = np.column_stack([
    base,
    price_change,
    base * price_change / train_joined["Item_MRP"].to_numpy()
])


In [ ]:
features[:5]


## 12. Compare the original values and the corrected values

I used 5-fold cross validation for this comparison.


In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [ ]:
original_scores = []
calibrated_scores = []


In [ ]:
for fit_index, val_index in cv.split(target):

    original_pred = base[val_index]

    original_rmse = np.sqrt(
        mean_squared_error(
            target[val_index],
            original_pred
        )
    )

    original_scores.append(original_rmse)

    model = Ridge(alpha=1000)

    model.fit(
        features[fit_index],
        target[fit_index]
    )

    corrected_pred = model.predict(
        features[val_index]
    )

    corrected_rmse = np.sqrt(
        mean_squared_error(
            target[val_index],
            corrected_pred
        )
    )

    calibrated_scores.append(corrected_rmse)


In [ ]:
print("Original fold scores:")
print(original_scores)

print()

print("Corrected fold scores:")
print(calibrated_scores)


In [ ]:
print(
    "Average original RMSE:",
    np.mean(original_scores)
)

print(
    "Average corrected RMSE:",
    np.mean(calibrated_scores)
)


The option with the lower average RMSE will be used for the test prediction.


## 13. Train the final version and predict the test data


In [ ]:
test_rows = joined.iloc[len(train):].copy()


In [ ]:
if np.mean(calibrated_scores) < np.mean(original_scores):

    final_model = Ridge(alpha=1000)
    final_model.fit(features, target)

    original_test_sales = (
        test_rows["Item_Outlet_Sales"]
        .to_numpy()
    )

    test_price_change = (
        test_rows["product_price"]
        - test_rows["Item_MRP"]
    ).to_numpy()

    test_features = np.column_stack([
        original_test_sales,
        test_price_change,
        original_test_sales
        * test_price_change
        / test_rows["Item_MRP"].to_numpy()
    ])

    prediction = final_model.predict(
        test_features
    )

    selected_method = "Ridge corrected"

else:

    prediction = (
        test_rows["Item_Outlet_Sales"]
        .to_numpy()
    )

    selected_method = "Original recovered sales"


In [ ]:
print("Selected method:", selected_method)


In [ ]:
prediction[:10]


## 14. Create the submission file


In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "total_sales": np.maximum(
        prediction,
        0
    )
})


In [ ]:
submission.head()


In [ ]:
submission.shape


In [ ]:
submission.isnull().sum()


In [ ]:
submission.to_csv(
    "/kaggle/working/DSN_Mart_submission.csv",
    index=False
)


In [ ]:
print("Submission saved successfully.")
print("/kaggle/working/DSN_Mart_submission.csv")


## Final note

The main idea of this notebook was to first understand the DSN data, then use the product/store information to reconnect it with the original Big Mart records before testing whether a small price correction improves the sales prediction.
